In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "badrex/kinyarwanda-speech-1000h",
    split="train",
    streaming=True
)

print(dataset)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/226 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/226 [00:00<?, ?it/s]

IterableDataset({
    features: ['audio_id', 'audio', 'audio_duration', 'gender', 'age_group', 'category', 'transcription'],
    num_shards: 226
})


## Step 1 — Remove the audio-related columns

In [2]:
text_dataset = dataset.select_columns([
    "audio_id",
    "category",
    "transcription"
])

print(text_dataset)

IterableDataset({
    features: ['audio_id', 'category', 'transcription'],
    num_shards: 226
})


## Step 2 — First, inspect only 20 transcripts

In [3]:
import pandas as pd

first_20 = list(text_dataset.take(20))

df_20 = pd.DataFrame(first_20)

pd.set_option("display.max_colwidth", None) 

display(df_20)

,audio_id,category,transcription
0,002t9cjMXzngNV2MuT6h,Financial Services,"Iri ni isiko rikorera kuri murandasi ricuruza ibirimo n'ibyo kurya, aha hari amandazi bagurisha amafaranga maganatatu mirongo itanu bikaba ari ibintu bishimishije umuntu ashobora kuyagura yifashishije ikoranabuhanga."
1,004XBLHMJjhPBkRDG58a,Health,"Siporo si izikorerwa mu mihanda cyangwa ahubatswe za jimu gusa, hari n'izo wakorera iwawe mu rugo, ku buriri kandi na zo zikagirira umumaro uhagije umubiri wawe."
2,00HDMjHbqepMJ0wBxGUf,Health,"Abagore batatu bahuriye kumeza barakina amakarita barishimye, harimo umugore wambaye agapira gatukura muto ufite imisatsi myiza y'umukara, harimo nundi mugore wambaye agapira k'umukara inyuma yifutse akandi k'umweru arashaje afite imisatsi irimo imvi."
3,00JDQTWfDo6L6ObZVvPx,Financial Services,Icyapa kinini kirekire kiriho ifoto y' umukecuru n' umusaza b' abazungu ni itangazo ryamamariza ikigo cy' ubwishingizi.
4,00MR06OHUElcSafypT1H,Government Services,"Imodoka nziza cyane itwara imizigo ibirayi, amakara, ibitoki n'ibindi, irimo irabisikana n'umumotari hirya ye hari indi y'umweru hakurya harizo bari gupakiramo imifuka y'imihondo myiza cyane icyeye kandi bagiye kujyana mu mahanga."
5,00MRhnmS1alSC2wvHCXj,Financial Services,"Abagore bakenyeye ibitenge, bahagaze, hari n'undi wicaye, hari umuryango ukinguye, hakaba inyuma yabo ibitebo biriho indimu ndetse n'izindi mbuto, hirya yaho hari amadegede."
6,00NdgshpAjFnSQVpykG3,Health,"Abana babanyeshuri benshi bari mu kigero cy'imyaka itanu, abakobwa n' abahungu bose bicaye hamwe, bamwe bambaye impuzankano y' ibara ry' umuhondo abandi biyambariye imipira isanzwe, bose baraseka barishimye bareba imbere hari umuntu urimo kubaganiriza abasekera."
7,00OBvWocum44e8hi3Yn6,Health,"Ikirango kiri mu mabara y'ubururu ndetse n'umweru n'umutuku, kikaba cyerekana uburyo imibereho y'umugore, umwana ndetse n'abantu muri rusange ihagaze."
8,00PyLrcpRkPGPNneKnLq,Health,"Umuntu ari ku rukarabiro, arimo arakaraba amazi meza asukuye, kugira ngo avane imyanda iri ku ntoki, yaba yikuye aho yiriwe ndetse n'ibyo yakozeho, bityo akomeze kugira ubuzima, bwiza kandi abashe kuramba."
9,00ShFmc33nbQuZBw1vfb,Government Services,Abantu benshi bari mu nama bicaye bari kumva ibyo bagezweho harimo umugabo wambaye umupira w'ubururu urimo urandika imbere ye hariho uducupa tw'amazi.


## Step 3 — Check what categories exist

In [4]:
# Take a larger temporary sample only to inspect categories
temp = list(text_dataset.take(5000))
temp_df = pd.DataFrame(temp)

print(temp_df["category"].value_counts(dropna=False))

category
Health                 1827
Financial Services     1636
Government Services    1537
Name: count, dtype: int64


In [5]:
shuffled_dataset = text_dataset.shuffle(
    seed=42,
    buffer_size=20000
)

temp = list(shuffled_dataset.take(10000))

temp_df = pd.DataFrame(temp)

print("Number of records:", len(temp_df))
print("\nCategories found:")
print(temp_df["category"].value_counts(dropna=False))

Number of records: 10000

Categories found:
category
Health                 3355
Government Services    3018
Financial Services     2980
Agriculture             458
Education               189
Name: count, dtype: int64


In [6]:
categories = [
    "Health",
    "Government Services",
    "Financial Services",
    "Agriculture",
    "Education"
]

samples = []

for category in categories:
    category_df = temp_df[
        (temp_df["category"] == category) &
        (temp_df["transcription"].notna())
    ]
    
    sampled = category_df.sample(
        n=100,
        random_state=42
    )
    
    samples.append(sampled)

sample_df = pd.concat(samples, ignore_index=True)

print("Total transcripts:", len(sample_df))
print()
print(sample_df["category"].value_counts())

Total transcripts: 500

category
Health                 100
Government Services    100
Financial Services     100
Agriculture            100
Education              100
Name: count, dtype: int64


In [7]:
sample_df.to_csv(
    "/kaggle/working/kinyarwanda_preliminary_sample_500.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [8]:
pd.set_option("display.max_colwidth", None)

manual_sample = sample_df[
    ["category", "transcription"]
].sample(
    30,
    random_state=10
)

display(manual_sample)

,category,transcription
151,Government Services,"Iki ni ikibuga kinini cyane, gitewemo ibyatsi bisa icyatsi hakaba harimo umuntu uri kugendamo, wambaye umupira usa ubururu n'ipantaro. Hirya hakaba hari ibyapa byinshi, byiganjem ubururu n'umweru."
424,Education,"Inyubako y'ubucuruzi ndetse harimo n'abantu, basohoye kamwe kaba karimo ibicuruzwa by'ibanze, kugira ngo uhageze wese, amenye icyahakorerwa kugira ngo agurire hafi."
154,Government Services,"Iyi modoka nini itwara abantu mu buryo bwa rusange, ikaba iparitse isize amabara y'umweru akaba ari abantu barimo, hirya hakaba n'indi modoka na yo imeze nka yo ziparitse zikaba zimeze nk'iziri ahantu muri gare zitegereje gutwara abagenzi."
190,Government Services,"Igikorwaremezo kirekire gikozwe mu bikoresho by'ibyuma kiriho insinga nyinshi cyane, cyifashishwa mu kugeza ndetse no kwegereza umuriro w'amashanyarazi ku baturage."
131,Government Services,"Abantu batatu ,abagabo babiri n'umugore umwe, umugabo umwe afashe mikoro , nk'aho ari gutanga igitekerezo . Naho umugore na we afite imbere y'iwe ikaye, umugabo nawe afite ikaye n'ikaramu. Ku mpande hateretseho amazi , inyuma yabo, hariho icyapa gisa ubururu,cyakishije mu nyuguti zisa umweru."
310,Agriculture,"Ubuhinzi bw'inyanya bukozwe ku buryo bwa kijyambere, bufasha abahinzi mu kubona umusaruro uhagije ukomoka ku buhinzi, ibi bikaba bibafasha mu kubona amafaranga ndetse n'ibyo kurya bibafasha mu kubaho mu buzima bwabo bwa buri munsi."
266,Financial Services,"Inyubako nziza ya komvesheni (convetion) igaragaza amabara agize idarapo yigihugu cyacu ariyo ubururu ,umuhondo n'icyatsi niheza cyane."
495,Education,"Inzu y'ishuri abana bigiramo, hakaba hanze hari idarapo ry'Igihugu, rikaba risakaje amabati y'ubururu, rikaba risize irangi ry'umweru ku bikuta, irindi ry'umukara hasi, hakaba inzugi zabo ni umukara, ibirahure bikaba umweru."
157,Government Services,"Hano ndahabona umuhanda wa kaburimbo ukuze hari dayihatsu ifite ibara ry'ubururu ipakiye ibyuma inyuma, hirya harimotari uhetse ndetse hari n'abagenzi bari kugenda."
282,Financial Services,"Ameza afite ibara ry'igitaka ariho terefone ngendanwa ifite ibara ry'umukara, na ekuteri ifite ibara ry'umukara hariho umuntu wambaye ishati y'amaboko maremare ifite ibara ry'igitaka irimo uturonko dufite ibara ry'icyatsi, hakaba hariho amafaranga ku meza afite amabara agiye atandukanye, hariho imibare yandikishije ibara ry'igitaka hariho amagambo yandikishijwe ibara ry'umweru inyubako ifite ibara ry'igitaka."


In [9]:
for category in categories:
    print("\n" + "=" * 80)
    print(category.upper())
    print("=" * 80)
    
    examples = sample_df[
        sample_df["category"] == category
    ][["transcription"]].sample(
        10,
        random_state=42
    )
    
    display(examples)


HEALTH


,transcription
83,"Umugore uri gukora imyitozo ngororamubiri, wambaye isengeri y'umutuku n'ipantaro y'umutuku, mu cyumba cyabugenewe gukoreramo imyitozo ngororamubiri, haragaragaramo kandi ibikoresho bikoreshwa mu gukora imyitozo ngororamubiri kandi uyu mugore yifashe ku mano, asoberekeranyije amatako yiwe kandi asa nk'aho araramye bigaragara ko ari gukwedura umugongo kugira ngo urusheho kugororoka neza."
53,"Abaganga bambaye imyenda y'akazi ifite ibara ry'ubururu bambaye udupfukamunwa. Bambaye bikwije ahantu hose. Barimo kureba umuntu wasamye wambaye imyenda y'umukara. Inyuma yabo hari itete, imeza iteretseho imiti, ndetse n'abandi bantu baje kwisuzumisha."
70,"Uruhurirane rw'imiti itandukanye ipfunyitse mu macupa ndetse no mu dupapuro twabugenewe, mu rwego rwo kwirinda ko yakangirika."
45,"Inzu nini ifite amadarapo abiri imbere yayo, ikaba ifite n'umuryango munini w'igipangu binjiriramo, ikaba ari inzu yifashishwa mu mibereho y'abaturage ya buri munsi kugira ngo yorohereze abaturage kubona serivisi zigiye zitandukanye."
44,"Umugore wambaye umupira, akenyeye igitenge akaba yiteze n'ikindi gitenge. Yazanye n'abandi bagore bagenzi be ku bitaro aho baje gushaka serivise, bakaba bakiriwe n'umuganga w'umugore aho ari kubapima ngo abe yabaha ubufasha bwo kubavura ndetse akabaha n'imiti."
39,Amababi atandukanye akoreshwa nk'imiti gakondo ndetse ikitabwaho n'abaganga b'inzobere kugira ngo bakoremo imiti ishobora guhabwa abarwayi bafite ibibazo by'indwara zitandukanye bitabweho umunsi ku munsi.
22,Umusaza uri gushishikariza abantu kurya gufata urukingo rw'indwara iterwa na virusi ituruka mu nkende kugira ngo babashe kuba bayirinda hakiri kare no kuba bayirwanya byihuse.
80,"Ni icyumba cy'abarwayi kwa muganga, harimo ibitanda bitandukanye ku rukuta hateye irangi ry'umuhondo kandi harimo na'masupanete, hari n'undi wicaye mu kagare yambaye umupira w'icyatsi n'abandi bicaye ku gitanda."
10,"Umwana w'inzobe, ufite imisatsi ku mutwe, wambaye umupira w'icyatsi ari kumwe n'umugabo ufite ubwanwa, wambaye itaburiya y'umweru."
0,"Siporo ni ingenzi ku buzima bwa buri munyagihugu wese, dore ko siporo ituma umubiri wacu ugira ubudahangarwa bukomeye cyane ku buryo hari indwara nyinshi cyane zitandukanye zidashobora gufata umubiri wacu, harimo nk'indwara y'umuvuduko w'amaraso, indwara zifata ubuhumekero, indwara zifata umutima ndetse n'izindi nyinshi cyane zitandukanye. Buri muntu wese rero nkaba namushishikariza gukora siporo, kuko ari igikorwa cy'ingenzi cyane ku buzima bwacu bwa buri munsi."



GOVERNMENT SERVICES


,transcription
183,Abantu benshi bicaye ku ntebe z'umukara zizengurutse ameza y'uruziga imbere yabo kumeza hatetseho amacupa ndetse n'ibirahuri by'amazi ku meza y'imbere hicaye abazungu babiri b'umugabo ndetse umugore bari kureba inyuma.
153,"Ibiti bibiri binyuraho insinga z'amashanyarazi, igiti kimwe cyikaba kiriho mubazi ebyiri inyuma y'ibiti by'amashanyarazi hakaba hari inzu ifite ibara ry'umuhondo n'icyapa kiriho gifite ibara ry'ubururu, umweru ndetse n'icyatsi."
170,Uyu ni umuhanda wo mu bwoko bwa kaburimbo usize amabara y'umukara n'uturongo tw'umweru ku ruhande ndetse no hagati hirya gato hari ibiti birebire bitanga umuyaga n'amahumbezi ku bari aho ngaho bose.
145,"Icyumba kirimo abantu batandukanye, bigaragara ko bari gutangiramo ubufasha ku bijyanye no gutora, kikaba kirimo umuntu bigaragara ko yaje gufashwa ngo abone uko atora, hakaba harimo inkuta ziriho ibitambaro biri mu ibara ry'icyatsi, umuhondo n'ubururu."
144,"Hano ndahabona abacamanza bari imbere bambaye iby'umutuku, nkahabona abashinjacyaha bari hino ndetse n'abavoka bambaye iby'umukara, hagati hari amabendera ya repubulika yu Rwanda, akaba ariyo akoreshwa mu kurahira kugira ngo umuntu yinjire mu nshingano ziremereye nkizi ngizi."
139,"Umupapa usa n'ukuze rwose, umusatsi yarawogoshe, ikote rye, ishati yanigirije, yifashe mu ntoki zahuye, imbere hari uducupa tw'amazi tubiri, ashobora kwifashisha igihe yumva aboberewe mu kanwa, akabasha kuba yahanyereza."
122,"Imbuga ngari y'igitaka, hubatsemo urugomero rw'amashanyarazi, rushinzwe kuyatunganya kugira ngo ruyohereze mu ngo zijye zitandukanye, zigize igihugu cyacu, amashanyarazi ni ingenzi mu buzima."
180,"Iki ni igikoresho cyifashishwa mu gukwirakwiza umuriro w'amashanyarazi, gikikijwe n'ibyatsi byinshi ndetse n'ibiti birebire ubona bitanga umuyaga n'amahumbezi ku begereye aho ngaho bose."
110,Abantu batanu bahagaze harimo abagabo batatu abagabo babiri b'ibumoso bambaye amakoti na za karuvati umwe yambaye ikotite ntakaruvati n'abagore babiri bitabiriye inama.
100,"Iyi ngiyi ni gereza, iki ngiki ni igipangu cya gereza, ahangaha niho bashyira abantu bagiye bemezwa ko bakoze ibyaha bigiye bitandukanye kugira ngo bakomeze babahe umuco banabagorore."



FINANCIAL SERVICES


,transcription
283,"Aha ngaha ni ahantu heza wasohokera, wafatira amashusho hari urwogero rwiza, ibitanda byiza byo kuryamaho mu gihe uvuye mu mu mazi, cyangwa se mu mu gihe utegereje kujyamo ukaba wota akazuba."
253,"Ni inyubako nziza cyane iherere imayange, ifite amakaro kandi iri kugurishwa. Ufite miliyoni cumi neshanu z'amafaranga y' u rwanda wayigura."
270,Muri karitsiye yacu harimo umucuruzi mushyashya yantangaje rwose. Ibaze ko yaje kandi mu iduka rye ntabwo ryari rinini agatangira kutwereka ibicuruzwa byose afite n'amafaranga bigura kandi twese turi kubyisomera turi no kubyirebera. Gusa yarafite amata aryoshye pe!
245,"Ibinyobwa bidasembuye bipanze neza mu kabati mu buryo bukurikiranye, harimo imitobe n'amazi, biri mu cyumba kimurikiwe n'amatara afashe mu gisenge, amurika neza."
244,"Akazu k'ikigo cy'itumanaho cya Emutiyeni kari ku muhanda kafunze, bugaragara ko kamaze igihe."
239,"Umudugudu urimo amazu yubatse mu bwoko butandukanye hari umuhanda urimo abamotari batwara abagenzi ndetse hari n'amapoto ariho insinga zitwara umuriro w'amashanyarazi muri uwo mudugudu, hari n'agace karimo ibikoresho byo mu nzu haba utubati intebe ndetse na matera."
222,"Agakoresho gakozwe mu mbaho bigaragara ko ari akameza gafiteho ububiko bugiye butandukanye, ku ruhande hagaragara umuryango uriho urugi rw'icyuma rusize irangi ry'ubururu ndetse n'akandi gakoresho kari munzu katagaragara neza."
280,"Abagore bicaye ahubatse neza mu ibara ry'umuhondo bikikijwe n'umutaka, ibyo byose ni iby'ikigo cy'itumanaho cya emutiyene. Abo bagore hari utukoresho imbere tw'umuhondo natwo duhari, ibyo ni ibyo bari kumurika. Ubwo bari mu bikorwa by'icyo kigo cya emutiyene"
210,"Abana ni bantu bagera kuri batatu bahagaze ahantu hamwe urabona ko bafite ibyishimo, higanjemo abagabo babiri n'umugore umwe bateruye ikintu kinini kimeze nk'urufunguzo mu biganza byabo."
200,"Inzu zifite ubusitani burimo ibyatsi ndetse n'uduti duto tutarakura, hari n'imbuga iriho ikigega cyubatse hejuru ku kazu k'amatafari ahiye, ikigega kirasa n'umukara kibika amazi."



AGRICULTURE


,transcription
383,"Abakora umwuga w'uburobyi batangaza ko imirimo bakora ibatunga mu buzima bwabo bwa buri munsi kuko batunga imiryango yabo ikindi kandi bagasagurira n'amasoko bityo bakabasha kubona amafaranga, ni byiza ko twese dusobanukirwa akamaro ko kuba twakora imyuga itandukanye kugira ngo tubashe kuba twakiteza imbere."
353,"Igitoki kiri mo inkingi, igitoki bakaba bakirya, cyitwa intude. Kikaba gitanga isosi imeze neza cyane."
370,"Inyama z'amafi bashishuye zokeje zikaba ari inyama bajya bafata amafi mabisi bakayashishura, ubundi bakazotsa kugira ngo abaguzi bitabagora barimo barazitononjoramo amagufa."
345,"Ubuvuzi bw'amatungo ni ubuvuzi buri kugenda butera imbere muri iki gihe bijyanye n'uko imiti yabwo iri kugenda yaguka ari myinshi, iyi miti iri guterwa n'uko hari ibyorezo byinshi biri kuza, bityo kugira ngo aborozi babirwanye bakifashisha imiti yabugenewe ivura amatungo."
344,"Ni mu ishyamba bigaragara ko hari ibiti birebire cyane, hari ikirere cy'ubururu, imbere harimo inyamaswa y'amahembe abiri asa naho avunitse, inyamaswa ifite ikintu iri kwitegereza bisa naho ifite ubwoba cyane."
339,"Igihingwa cy'urubuga akaba ari igihingwa giteye mu murima, aho gitatanye n'ibindi, gisa n'aho ari kimwe mu murima, iki gihingwa kikaba cyifashishwa mu kuzamura imirire myiza y'abantu bakuru ndetse n'abana."
322,"Umurima basaruyemo ibigori, uteye hafi y'urutoki, amago akikije ugwo muririma, hari ibitoki, ibiti n'ibindi bintu bitandukanye, bituma uwo murima ugira akayaga keza."
380,"Ubworozi ni bwiza kuko bituma umuntu atera imbere, haba mubereho, bukungu, ndetse n'ibindi. Rero mu bworozi cyane cyane inkoko, ingurube, zitanga ifumbire ndetse n'ibindi."
310,"Ubuhinzi bw'inyanya bukozwe ku buryo bwa kijyambere, bufasha abahinzi mu kubona umusaruro uhagije ukomoka ku buhinzi, ibi bikaba bibafasha mu kubona amafaranga ndetse n'ibyo kurya bibafasha mu kubaho mu buzima bwabo bwa buri munsi."
300,"Igihingwa k'ikawa ni igihingwa kivamo umusaruro utubutse kandi kikaba gitanga amafaranga menshi.Abahinzi barashishikarizwa guhinga igihingwa k'ikawa kugira ngo babashe kubona amafaranga menshi,babashe kwiteza imbere bityo ubukene bubashe kugabanuka.Ni byiza ko tugomba gusigasira igihingwa k'ikawa aho tukibonye hose kubera ko ari ingenzi mu buzima bwacu ndetse kikaba ari n'ubukungu mu Gihugu cy'u Rwanda."



EDUCATION


,transcription
483,"Igitabo kirimo ibara ry'icyatsi ndetse n'umuhondo werurutse, gishushanyijeho abana babiri b'abahungu, umwe yambaye ishati y'icyatsi, undi yambaye umupira wa orange. Bari gusoma igitabo cy'ikinyarwanda, igitabo cy'umunyeshuri umwaka wa mbere w'amashuri abanza."
453,"Imbuga nziza cyane, irimo ikicumba cy'abana, kiri mu mabara y'icyatsi ubururu n'umuhondo, ikikijwe n'urugo rw'amatafari, ahiye inyuma yaho hari ibiti bigandaye."
470,"Mu muhanda w'igitaka mutoya, harimo haranyuramo abakobwa babiri aribo bagaragara cyane, imbere yabo harimo haragenda abandi, ku mpande z'uyu muhanda hari ibyatsi."
445,"Umunyeshuri w'umukobwa ndetse n'uw'umuhungu bose bafashe kuri Bibiliya, barimo barasomera ijambo ry'Imana bagenzi babo ndetse n'abandi babateze amatwi, inyuma yabo hari abandi bana bari kwandika ibyo bize."
444,"Mu mashuri menshi ya pirimeri ndetse n'amashuri ya segonderi tugiye dusanga harimo ibibaho abarimu bifashisha bandikaho kugirango ngo abanyeshuri bakurikirane, aha ngaha ni muri pirimeri na ho hakaba hari ikibaho cy'umukara abarimu bandikaho, hasi hameze nk'aho hadasimase ahubwo ari ibitaka."
439,"Ibikinisho byiza bifite amabara atandukanye bambitse utwenda dutandukanye, ibintu abana bakiri batoya biga mu mashuri y'incuke bareba bakishima bakizihirwa."
422,"Umwarimu ari mu ishuri, arimo kwigisha umunyeshuri we, uko yandika imibare amukosora umwarimu yambaye itaburiya yabugenewe kugira ngo ingwa zitamutumukira, arimo yereka umunyeshuri we neza, uko bandika ibyo ari kumwigisha."
480,"Abanyeshuri bicaye ndetse hari n'abandi bahagaze, bambaye inifomo zisa ndetse bicaye muri jaride munsi y'ibiti."
410,Ikigo cy'amashuri abanza hari ishuri ririmo abanyeshuri bari kwiga mwarimu wabo akaba yambaye ijipo ngufi irimo utudomo tw'umweru n'umukara ndetse n'itaburiya y'umweru akaba ari kubaha imyitozo yo gukora.
400,"Gusubirishamo abana amasomo, ntabwo ari iby'umubyeyi w'umudamu gusa, ahubwo n'umugabo yabikora ndetse byaba akarusho, agafatanya n'umubyeyi, mu gusubirishamo abana amasomo yabo kugira ngo babyumve neza."


In [10]:
import re
import pandas as pd

def starts_with_capital(text):
    text = str(text).strip()
    for char in text:
        if char.isalpha():
            return char.isupper()
    return False

def ends_with_punctuation(text):
    text = str(text).strip()
    return bool(re.search(r'[.!?]$', text))

def has_punctuation(text):
    return bool(re.search(r'[.,!?;:]', str(text)))

audit_df = sample_df.copy()

audit_df["starts_with_capital"] = (
    audit_df["transcription"].apply(starts_with_capital)
)

audit_df["ends_with_punctuation"] = (
    audit_df["transcription"].apply(ends_with_punctuation)
)

audit_df["has_punctuation"] = (
    audit_df["transcription"].apply(has_punctuation)
)

results = audit_df.groupby("category").agg(
    total=("transcription", "size"),
    starts_capital=("starts_with_capital", "sum"),
    ends_punctuation=("ends_with_punctuation", "sum"),
    has_punctuation=("has_punctuation", "sum")
)

results["starts_capital_%"] = (
    results["starts_capital"] / results["total"] * 100
).round(1)

results["ends_punctuation_%"] = (
    results["ends_punctuation"] / results["total"] * 100
).round(1)

results["has_punctuation_%"] = (
    results["has_punctuation"] / results["total"] * 100
).round(1)

display(results)

,total,starts_capital,ends_punctuation,has_punctuation,starts_capital_%,ends_punctuation_%,has_punctuation_%
category,,,,,,,
Agriculture,100,100,100,100,100.0,100.0,100.0
Education,100,99,99,100,99.0,99.0,100.0
Financial Services,100,100,97,99,100.0,97.0,99.0
Government Services,100,99,99,99,99.0,99.0,99.0
Health,100,100,100,100,100.0,100.0,100.0


In [11]:
from collections import defaultdict

forms = defaultdict(set)
examples = defaultdict(list)

for _, row in sample_df.iterrows():
    text = str(row["transcription"])
    
    words = re.findall(
        r"[^\W\d_]+(?:['’][^\W\d_]+)*",
        text,
        flags=re.UNICODE
    )
    
    # Skip first word because sentence beginnings are naturally capitalized
    for word in words[1:]:
        key = word.lower()
        forms[key].add(word)
        
        if len(examples[key]) < 5:
            examples[key].append(text)

inconsistent = []

for word, variants in forms.items():
    has_upper = any(v[0].isupper() for v in variants if v)
    has_lower = any(v[0].islower() for v in variants if v)
    
    if has_upper and has_lower:
        inconsistent.append({
            "word": word,
            "variants": ", ".join(sorted(variants)),
            "examples": examples[word]
        })

inconsistent_df = pd.DataFrame(inconsistent)

print("Potential capitalization inconsistencies:",
      len(inconsistent_df))

display(inconsistent_df.head(30))

Potential capitalization inconsistencies: 77


,word,variants,examples
0,ni,"Ni, ni","[Siporo ni ingenzi ku buzima bwa buri munyagihugu wese, dore ko siporo ituma umubiri wacu ugira ubudahangarwa bukomeye cyane ku buryo hari indwara nyinshi cyane zitandukanye zidashobora gufata umubiri wacu, harimo nk'indwara y'umuvuduko w'amaraso, indwara zifata ubuhumekero, indwara zifata umutima ndetse n'izindi nyinshi cyane zitandukanye. Buri muntu wese rero nkaba namushishikariza gukora siporo, kuko ari igikorwa cy'ingenzi cyane ku buzima bwacu bwa buri munsi., Inzu y'imiti irimo abantu babiri ni imiti gakondo iri mu macupa atandukanye, hari amacupa apfundikije imipfundikizo y'umweru, hari andi y'umuhondo, andi arasa icyatsi, andi yenda gusa umutuku, hirya yaho hari firigo y'umweru yegeranye na muganga., Ibikoresho bigezweho byifashishwa bari gutunganya umuti w'amenyo, uyu twita korogate, ni umuti utuma amenyo agira ubuzima bwiza kandi akaba akomeye kandi asa neza., Umwana ari gukaraba mu ntoki no mu maso, ari munsi ya tiyo irimo kuvamo amazi, ni byiza gukaraba kandi ugakaraba neza, ukoresheje amazi meza n'isabune., Icupa ryiza rirerire kandi ry'umweru ririho akapa k'umukara kariho amagambo yanditse mu ibara ry'umweru aha ngaha rero uyu ni umwe mu muti abantu bakora siporo bifashisha kuko bakandaho maze bakirinda mikorobe.]"
1,ku,"Ku, ku","[Siporo ni ingenzi ku buzima bwa buri munyagihugu wese, dore ko siporo ituma umubiri wacu ugira ubudahangarwa bukomeye cyane ku buryo hari indwara nyinshi cyane zitandukanye zidashobora gufata umubiri wacu, harimo nk'indwara y'umuvuduko w'amaraso, indwara zifata ubuhumekero, indwara zifata umutima ndetse n'izindi nyinshi cyane zitandukanye. Buri muntu wese rero nkaba namushishikariza gukora siporo, kuko ari igikorwa cy'ingenzi cyane ku buzima bwacu bwa buri munsi., Siporo ni ingenzi ku buzima bwa buri munyagihugu wese, dore ko siporo ituma umubiri wacu ugira ubudahangarwa bukomeye cyane ku buryo hari indwara nyinshi cyane zitandukanye zidashobora gufata umubiri wacu, harimo nk'indwara y'umuvuduko w'amaraso, indwara zifata ubuhumekero, indwara zifata umutima ndetse n'izindi nyinshi cyane zitandukanye. Buri muntu wese rero nkaba namushishikariza gukora siporo, kuko ari igikorwa cy'ingenzi cyane ku buzima bwacu bwa buri munsi., Siporo ni ingenzi ku buzima bwa buri munyagihugu wese, dore ko siporo ituma umubiri wacu ugira ubudahangarwa bukomeye cyane ku buryo hari indwara nyinshi cyane zitandukanye zidashobora gufata umubiri wacu, harimo nk'indwara y'umuvuduko w'amaraso, indwara zifata ubuhumekero, indwara zifata umutima ndetse n'izindi nyinshi cyane zitandukanye. Buri muntu wese rero nkaba namushishikariza gukora siporo, kuko ari igikorwa cy'ingenzi cyane ku buzima bwacu bwa buri munsi., Umusore ari gukora siporo, yarambuye ukuboko kwe n'ukuguru kwe arakuzamura, ukuboko kwe agukoza ku kuguru, ukundi kuguru kurahagaze ari kugorora ingingo ze mu rwego rwo kwirinda indwara zimwe na zimwe ziterwa no kudakora siporo., Umwana wambaye ikabutura itukura ndetse n'ishati y'amaboko maremare, ateze umunwa ku mugezi aho ari kunywa amazi andi mazi asagutse ku ntoki akagwa mu ndobo ari na yo yaje kuvomeramo.]"
2,buri,"Buri, buri","[Siporo ni ingenzi ku buzima bwa buri munyagihugu wese, dore ko siporo ituma umubiri wacu ugira ubudahangarwa bukomeye cyane ku buryo hari indwara nyinshi cyane zitandukanye zidashobora gufata umubiri wacu, harimo nk'indwara y'umuvuduko w'amaraso, indwara zifata ubuhumekero, indwara zifata umutima ndetse n'izindi nyinshi cyane zitandukanye. Buri muntu wese rero nkaba namushishikariza gukora siporo, kuko ari igikorwa cy'ingenzi cyane ku buzima bwacu bwa buri munsi., Siporo ni ingenzi ku buzima bwa buri munyagihugu wese, dore ko siporo ituma umubiri wacu ugira ubudahangarwa bukomeye cyane ku buryo hari indwara nyinshi cyane zitandukanye zidashobora gufata umubiri wacu, harimo nk'indwara y'umuvuduko w'amaraso, indwara zifata ubuhumekero, indwara zifata umutima ndetse n'izindi nyinshi cyane zitandukanye. Buri muntu wese rero nkaba namushishikar

In [12]:
# Transcripts that do not start with a capital letter
print("NOT STARTING WITH CAPITAL:")
display(
    audit_df.loc[
        ~audit_df["starts_with_capital"],
        ["category", "transcription"]
    ]
)

# Transcripts that do not end with . ! or ?
print("\nNO ENDING PUNCTUATION:")
display(
    audit_df.loc[
        ~audit_df["ends_with_punctuation"],
        ["category", "transcription"]
    ]
)

# Transcripts with none of the punctuation characters checked
print("\nNO PUNCTUATION:")
display(
    audit_df.loc[
        ~audit_df["has_punctuation"],
        ["category", "transcription"]
    ]
)

NOT STARTING WITH CAPITAL:


,category,transcription
135,Government Services,umugabo urimo kubazwa ibibazo bigiye bitandukanye arimo kubisubiza afite akanyamuneza ndetse n'umutima ucyeke ibyo bigaragaza yuko atarimo kubeshya mu byo ari kuvuga byose.
471,Education,"bigara ko ari ababyeyi babiri bari kurera abana mu mashuri matoya bakunze kwita nasari, nkuko mubibona muri kubona bari kubafasha mu bijyanye no kubagaburira kugira ngo baze gukomeza kwiga nta kibazo cy'ubuzima bafite."



NO ENDING PUNCTUATION:


,category,transcription
176,Government Services,Ni ihuriro ry'imihanda myinshi ririmo ibimenyetso byo mu muhanda yambukirwaho nabanyamaguru harimo imodoka Ndetse n'ibindi binyabiziga harimo n'ibimenyetso byo muhanda bimurika n'inyubako zikikije iyo mihanda na esitasiyo ya esanse
273,Financial Services,"Akabari gafite mo imbere ndetse kakagira no hanze hagaragara inebe zitandukanye nziza n'ameza, hakaba hari abantu batandukanye baje kwiyakira mu kabari aho batangira serivise zitandukanye zirimo ibyo kurya n'ibyo kunywa ku bantu baje bahagana"
280,Financial Services,"Abagore bicaye ahubatse neza mu ibara ry'umuhondo bikikijwe n'umutaka, ibyo byose ni iby'ikigo cy'itumanaho cya emutiyene. Abo bagore hari utukoresho imbere tw'umuhondo natwo duhari, ibyo ni ibyo bari kumurika. Ubwo bari mu bikorwa by'icyo kigo cya emutiyene"
286,Financial Services,Aho banywera inzoga hicaye abantu benshi ku ntebe z'ubwoko butandukanye ndetse n'ameza afite igice cyo hejuru gikoze mu rubaho harimo urumuri rukeye rw'amatara make y'umuhondo
476,Education,"Amashuri ni ahantu abantu bajya kwiga bagakura ubumenyi butandukanye. Ubumenyi bubafasha kwihangira umurim,o ubwo baba bafata ibyo bize bakabikoresha mu buzima bwabo bwa buri munsi busanzwe"



NO PUNCTUATION:


,category,transcription
176,Government Services,Ni ihuriro ry'imihanda myinshi ririmo ibimenyetso byo mu muhanda yambukirwaho nabanyamaguru harimo imodoka Ndetse n'ibindi binyabiziga harimo n'ibimenyetso byo muhanda bimurika n'inyubako zikikije iyo mihanda na esitasiyo ya esanse
286,Financial Services,Aho banywera inzoga hicaye abantu benshi ku ntebe z'ubwoko butandukanye ndetse n'ameza afite igice cyo hejuru gikoze mu rubaho harimo urumuri rukeye rw'amatara make y'umuhondo


In [13]:
import re
import pandas as pd

check_df = sample_df.copy()

# Whitespace / punctuation problems
check_df["multiple_spaces"] = check_df["transcription"].str.contains(
    r"\s{2,}", regex=True
)

check_df["space_before_punctuation"] = check_df["transcription"].str.contains(
    r"\s+[,.!?;:]", regex=True
)

check_df["missing_space_after_punctuation"] = check_df["transcription"].str.contains(
    r"[,.!?;:][A-Za-zÀ-ÖØ-öø-ÿ]",
    regex=True
)

check_df["repeated_punctuation"] = check_df["transcription"].str.contains(
    r"([.!?,;:])\1+",
    regex=True
)

# Apostrophes
check_df["straight_apostrophe"] = check_df["transcription"].str.contains(
    "'", regex=False
)

check_df["curly_apostrophe"] = check_df["transcription"].str.contains(
    "’", regex=False
)

check_df["space_after_apostrophe"] = check_df["transcription"].str.contains(
    r"['’]\s+",
    regex=True
)

check_df["space_before_apostrophe"] = check_df["transcription"].str.contains(
    r"\s+['’]",
    regex=True
)

columns_to_check = [
    "multiple_spaces",
    "space_before_punctuation",
    "missing_space_after_punctuation",
    "repeated_punctuation",
    "straight_apostrophe",
    "curly_apostrophe",
    "space_after_apostrophe",
    "space_before_apostrophe"
]

summary = []

for col in columns_to_check:
    count = check_df[col].sum()
    summary.append({
        "check": col,
        "count": count,
        "percentage": round(count / len(check_df) * 100, 2)
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

/tmp/ipykernel_58/2992414011.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  check_df["repeated_punctuation"] = check_df["transcription"].str.contains(


,check,count,percentage
0,multiple_spaces,38,7.6
1,space_before_punctuation,15,3.0
2,missing_space_after_punctuation,25,5.0
3,repeated_punctuation,2,0.4
4,straight_apostrophe,442,88.4
5,curly_apostrophe,2,0.4
6,space_after_apostrophe,4,0.8
7,space_before_apostrophe,0,0.0


In [14]:
issue_columns = [
    "multiple_spaces",
    "space_before_punctuation",
    "missing_space_after_punctuation",
    "repeated_punctuation",
    "curly_apostrophe",
    "space_after_apostrophe"
]

for issue in issue_columns:
    flagged = check_df[check_df[issue]]

    print("\n" + "=" * 80)
    print(issue.upper())
    print("Count:", len(flagged))
    print("=" * 80)

    display(
        flagged[
            ["category", "transcription"]
        ].head(20)
    )


MULTIPLE_SPACES
Count: 38


,category,transcription
9,Health,"Umugore wasamye yambaye iherena ku gutwi kw'ibumoso, kandi hari umuganga umuri iruhande, uri kumukura amenyo wambaye ga zifite ibara ry'umweru ndetse n'ishati y'amaboko magufi."
21,Health,"Umuntu wambaye umupira w'umukara w'amaboko maremare, uri guhingukwamo n'undi w'umweru, ahinnye intoki, afite agatambaro k'ibara ry'umweru, ari kwihanagura mu maso, ariko kandi yambaye n'ingofero y'ibara ry'umukara y'ubwoya."
67,Health,Abanyeshuri bambaye impuzankano barimo kunywa amazi meza ku kigega bakoresheje ibikombe kunywa amazi akaba arimwe mu nama zitangwa n'impuguke mu byubuvuzi ko byibuze umuntu yakabaye anywa litiri eshatu ku munsi kugira ngo agire ubuzima bwiza kuko amazi ari ingenzi cyane mu mubiri kandi afasha mu bikorwa bitandukanye nko gusohora imyanda ndetse n'ibindi.
88,Health,"Ikibuga cy'umupira w'amaguru kiriho abantu benshi, bamwe bari kure, abandi bari hafi babiri, umwe aryamye hasi areba hejuru, yarambuye akaguru kamwe, akandi yaguhinnye mu ivi, uwundi muntu apfukamye imbere ye n'ivi rimwe amaboko ye yombi ayafatisha akaguru k'uwo muntu yazamuye, hirya yabo hari umupira bakinisha amaguru."
101,Government Services,"Ahari mu muhanda hari urujya n'uruza rw'abantu, ndetse n'ibinyabiziga bitandukanye, harimo amapikipiki atwaye abantu, ndetse n'amamodoka hari kandi inzu nini ifite amadirishya y'icyatsi irimo ubucuruzi butandukanye bw'ibicuruzwa bitandukanye ndetse n'imiti, n'ubucuruzi bw'imiti."
109,Government Services,Umudamu urimo gutanga inbwiruhame aho uwo mudamu afite indangururamajwi mu ntoki yo kwifashisha hakaba hari abantu bamukurikiye bicaye mu busitani bw'ikibuga.
111,Government Services,Umunara muremure w'itumanaho wifashishwa mu guhereza amakuru abaturage bawuturiye ndetse n'abo hirya yawo. Hubakishijwe ibyuma by'umutuku ndetse n'umweru hafi aho hari akazu abazamu bifashisha bawurinda.
127,Government Services,"Abantu benshi bicaye bakaba bicaye mu nama harimo abagabo ndetse n'abagore, bakaba bateze amatwi bari mu nzu nini ifite ibirahuri bibonerana, imiryango ikaba isa umukara hakaba arimo abazungu ndetse n'abirabura, hakaba hari ibiti biteye muri iyo nzu biri mu mavazi."
131,Government Services,"Abantu batatu ,abagabo babiri n'umugore umwe, umugabo umwe afashe mikoro , nk'aho ari gutanga igitekerezo . Naho umugore na we afite imbere y'iwe ikaye, umugabo nawe afite ikaye n'ikaramu. Ku mpande hateretseho amazi , inyuma yabo, hariho icyapa gisa ubururu,cyakishije mu nyuguti zisa umweru."
135,Government Services,umugabo urimo kubazwa ibibazo bigiye bitandukanye arimo kubisubiza afite akanyamuneza ndetse n'umutima ucyeke ibyo bigaragaza yuko atarimo kubeshya mu byo ari kuvuga byose.



SPACE_BEFORE_PUNCTUATION
Count: 15


,category,transcription
30,Health,"Imbere mu bitaro icyumba cyabugenewe gisuzumirwama kikanavurirwamo amaso harimo ibikoresho bitandukanye nk'intebe bicaramo umurwayi ugiye kuvurwa ,intebe yicaraho muganga urimo kumuvura ndetse n'ameza ateretseho mudasobwa umuganga akoreshe mu mugihe cyo kuvura umurwayi harimo ibindi bikoresho by'imitako bigaragaza ko iki cyumba aricyo kivurirwamo amaso."
130,Government Services,"Kaburimbo irimo imirongo y'umweru imwe igaragaza aho abanyamaguru banyura mu mugihe bari kwambuka ,harimo n'ibiti bitandukanye bifite amababi y'icyatsi harimo n'ikinyabiziga cy'pikipiki cy'icaho n'umuntu wambaye ijire y'umuhondo n'ubururu."
131,Government Services,"Abantu batatu ,abagabo babiri n'umugore umwe, umugabo umwe afashe mikoro , nk'aho ari gutanga igitekerezo . Naho umugore na we afite imbere y'iwe ikaye, umugabo nawe afite ikaye n'ikaramu. Ku mpande hateretseho amazi , inyuma yabo, hariho icyapa gisa ubururu,cyakishije mu nyuguti zisa umweru."
161,Government Services,"Imodoka iparitse muri kaburimbo ,umuhanda w'umukara hari ibiti hari n'icyapa kiriho amabara y'umutuku ndetse n'umweru hari abantu bari kwambuka umuhanda n'abamotari bari mumuhanda bahagaze bategereje ko bambuka umuhanda ."
266,Financial Services,"Inyubako nziza ya komvesheni (convetion) igaragaza amabara agize idarapo yigihugu cyacu ariyo ubururu ,umuhondo n'icyatsi niheza cyane."
317,Agriculture,Ifi yokeje imaze gushya ariko ikiri ku cyokezo yasizwe ibirungo irasa neza . I ruhande rwaho hariho ibirungo biyiherekeza iyo bayijyaniye ugiye kuyirya kugira ngo arye aryoherwe birushijeho.
334,Agriculture,"Aka ni akanyoni gasa nkaho kari gukubita ku giti kugira ngo gakoremo icyari, kagizwe n'amabara umweru, umukara ,umutuku ndetse n'ubwoya bwiza bwera de kerade."
340,Agriculture,"Aha hagaragara amashanyarazi ,mu masambu ahingwamo ibihingwa, byo kurya ngandurarugo ndetse n'ibihingwa ngandurabukungu, hari uruhande ruriho umuhanda."
357,Agriculture,"Ibihingwa by'ingandurarugo : ibijumba ,amateke,imyumbati bidufashaga kurya tukihaza mu biribwa tugasagurira n'amasoko ntitugire inzara kuko tuba twarabyihingiye tukaja kugurisha tukagura n'ibindi tudafite."
404,Education,"Abana biga mu mashuri abanza bakenera ibikoresho bitandukanye birimo: amakaramu yo kwandikisha,amakaye yo kwandikamo, amarate yo gushushanya ,harimo imikasi yo guca impapuro zikoreshwa mu gushushanya hakabamo n'ikaramu z'igiti zikoreshwa mu gushushanya kugira ngo hatabaho kwibeshya."



MISSING_SPACE_AFTER_PUNCTUATION
Count: 25


...

,category,transcription
123,Government Services,"Inyubako ifite imiryango myinshi. Hamwe hakoreramo iduka ricuruza amatelefone, ivuriro, aho batangira ibikoresho by'ikoranabuhanga, imodoka, ibiti..."
381,Agriculture,"Hariho ibihingwa duhinga, urugero nk'ikawa, ikawa tuyihinga tugamije kubona amafaranga ukayihinga ugamije kuyigemura mu masoko menshi agiye atandukanye. Iyo kawa rero iyo wayiseho ikawa itanga umusaruro mwinshi cyane kubera ko igira amafaranga igira amafaranga abahinzi bayo rero nabashikariza gukoresha imborera nyinshi bakanazitaho kugira ngo...."



CURLY_APOSTROPHE
Count: 2


,category,transcription
37,Health,"Iki ni ikirango cy’ikigo cy’igihugu gishinzwe gutanga amaraso. Ikigo cy’igihugu gishinzwe gutanga amaraso, gifite akazi gakomeye ko gushishikariza abantu b’ingeri zitandukanye gutanga amaraso. Natwe tuba tugomba gutanga amaraso kuko iyo utanze amaraso, uba utanze ubuzima. Bityo rero, natwe tuba tugomba gufasha abantu dutanga amaraso."
95,Health,"Kwa muganga, umuganga ari kwita ku murwayi, har’ikintu amufatishije mu ntoki kimeze nk'itoroshi, uyu murwayi yambaye ishati y'ubururu, umuganga yambaye itaburiya y'umweru. Uyu murwayi bamwicaje mu gatebe biragaragara ko ari kugenda avuga ahantu ari kubabara."



SPACE_AFTER_APOSTROPHE
Count: 4


,category,transcription
152,Government Services,"Uyu ni umuhanda munini kandi mwiza wo mu bwoko bwa kaburimbo, usize amabara y'umukara ndetse n'uturongo tw'umweru kuruhande ndetse no hagati, hirya gato har' ibiti birebire ubona bitanga umuyaga ndetse n'amahumbezi sibyo gusa hari n'ikinyabiziga kiri kugendamo."
252,Financial Services,"Inyubako yubatswe mu buryo bugezweho, ifite ibikoresho mu nzu, iherereye mu mujyi wa Kigali muri Kanombe, igurishwa miliyoni ijana na cumi n'eshanu z'amafaranga y' u Rwanda."
253,Financial Services,"Ni inyubako nziza cyane iherere imayange, ifite amakaro kandi iri kugurishwa. Ufite miliyoni cumi neshanu z'amafaranga y' u rwanda wayigura."
385,Agriculture,"Iyi ni inzira yo mucaro y'amaguru itanyuragamo imodoka.Inyuragamo amagare n'abantu bari kugendesha amaguru kubera ko ni ntoya, iruhande rwayo gariho umurima gw' ibirayi."


In [16]:
domain_percentages = (
    check_df
    .groupby("category")[issue_columns]
    .mean() * 100
).round(1)

display(domain_percentages)

,multiple_spaces,space_before_punctuation,missing_space_after_punctuation,repeated_punctuation,curly_apostrophe,space_after_apostrophe
category,,,,,,
Agriculture,7.0,4.0,9.0,1.0,0.0,1.0
Education,4.0,6.0,7.0,0.0,0.0,0.0
Financial Services,8.0,1.0,4.0,0.0,0.0,2.0
Government Services,15.0,3.0,4.0,1.0,0.0,1.0
Health,4.0,1.0,1.0,0.0,2.0,0.0


## Preliminary Findings: Kinyarwanda Track B Dataset

A preliminary inspection was conducted on a balanced sample of **500 Kinyarwanda transcripts** from the Track B dataset. The sample included **100 transcripts from each of the five domains**:

- Health
- Agriculture
- Education
- Government Services
- Financial Services

Overall, the dataset was found to be **highly consistent in sentence-level punctuation and capitalization**. Most transcripts started with a capital letter and ended with appropriate punctuation. However, several smaller formatting inconsistencies were identified within the transcripts.

### Main Issues Identified

| Issue | Percentage | Observation |
|---|---:|---|
| Multiple spaces | 7.6% | Most common formatting issue |
| Missing space after punctuation | 5.0% | Some punctuation marks were immediately followed by another word |
| Space before punctuation | 3.0% | Unnecessary spaces appeared before punctuation marks |
| Missing sentence-final punctuation | 1.0% | Some transcripts did not end with `.`, `?`, or `!` |
| Space after apostrophe | 0.8% | Cases such as unnecessary spacing after `'` were observed |
| Sentence starts with lowercase | 0.4% | A small number of transcripts began with lowercase letters |
| Repeated punctuation | 0.4% | A few cases contained repeated punctuation marks |
| Curly apostrophe `’` | 0.4% | A small number used a different apostrophe representation |

### Capitalization and Sentence-Level Punctuation

Sentence-level capitalization was generally consistent:

- **99.6%** of transcripts started with a capital letter.
- **99.0%** ended with sentence-final punctuation.
- **99.6%** contained punctuation.

The few exceptions were confirmed to be genuine annotation inconsistencies rather than valid linguistic cases.

Manual inspection also identified some internal capitalization problems, such as capital letters appearing unnecessarily in the middle of sentences or words that appeared to require capitalization being written in lowercase.

### Whitespace and Punctuation Formatting

Whitespace-related inconsistencies were more common than sentence-level capitalization problems.

The most notable issue was **multiple spaces**, affecting **7.6%** of the sample. Government Services had the highest occurrence, with multiple spaces appearing in **15%** of its sampled transcripts.

Other formatting issues included spaces before punctuation, missing spaces after punctuation, and misplaced punctuation inside words.

An example of malformed punctuation observed during manual inspection was:

```text
umurim,o

## SECOND DATASET

In [17]:
from datasets import load_dataset

cv = load_dataset(
    "fsicoli/common_voice_19_0",
    "rw",                  # Kinyarwanda
    split="train",
    streaming=True
)

print(cv)

README.md: 0.00B [00:00, ?B/s]

common_voice_19_0.py: 0.00B [00:00, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found common_voice_19_0.py